# Uncertainty Analysis: Pitch Angle
This notebook analyzes the impact of camera pitch angle variations on slope estimation accuracy.
It includes:
1.  **Data Preparation**: Generating artificially pitched panoramas from original data.
2.  **Image Transformation**: Converting pitched panoramas to perspective views.
3.  **Result Analysis**: Comparing estimated slopes against ground truth.
4.  **Visualization**: Plotting uncertainty and data availability.

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import concurrent.futures
from pathlib import Path
from tqdm import tqdm
from zensvi.transform import ImageTransformer

# Plotting settings
plt.rcParams.update({'font.size': 14})
plt.rcParams['font.family'] = 'Arial'

In [ ]:
# Configuration & Paths
BASE_DIR = 'uncertainty_analysis'

# Input Data
VALIDATION_SAMPLES_PATH = os.path.join(BASE_DIR, 'subsamples_result.gpkg')
SOURCE_PANOS_DIR = os.path.join(BASE_DIR, 'panoramas')

# Working Directories
DEGRADATION_DIR = os.path.join(BASE_DIR, 'pitch')
PITCHED_PANOS_DIR = os.path.join(DEGRADATION_DIR, 'pitch-panos')
TRANSFORMED_DIR = os.path.join(DEGRADATION_DIR, 'pitch-test')

# Output
FIGURE_SAVE_PATH = os.path.join(BASE_DIR, 'pitch_uncertainty.png')

# Parameters
PITCH_ANGLES = list(range(-30, 31, 5))

In [ ]:
# Load validation samples
subsamples = gpd.read_file(VALIDATION_SAMPLES_PATH)
display(subsamples.head())

In [ ]:
def pitch_rotate_equirect(img, pitch_deg):
    """
    Rotate an equirectangular panorama around the pitch axis (x-axis).
    pitch_deg: Positive for looking up, negative for looking down.
    """
    h, w = img.shape[:2]
    j, i = np.meshgrid(np.arange(w), np.arange(h))
    lon = (j / w) * 2 * np.pi - np.pi
    lat = (i / h) * np.pi - (np.pi / 2)

    # Spherical coordinates
    x = np.cos(lat) * np.cos(lon)
    y = np.sin(lat)
    z = np.cos(lat) * np.sin(lon)
    vec = np.stack([x, y, z], axis=-1)

    # Rotation matrix for pitch (around Z-axis in this coordinate system setup or X depending on definition)
    # Based on provided code, it rotates around Z-axis of the vector space which corresponds to pitch in pano
    pitch = np.deg2rad(pitch_deg)
    Rz = np.array([
        [ np.cos(pitch), -np.sin(pitch), 0],
        [ np.sin(pitch),  np.cos(pitch), 0],
        [             0,              0, 1]
    ], dtype=np.float32)

    vec_rot = vec @ Rz.T

    # Reproject to lon, lat
    x2, y2, z2 = vec_rot[...,0], vec_rot[...,1], vec_rot[...,2]
    lat2 = np.arcsin(np.clip(y2, -1, 1))
    lon2 = np.arctan2(z2, x2)

    # Map to pixel coordinates
    j2 = (lon2 + np.pi) / (2 * np.pi) * w
    i2 = (lat2 + np.pi/2) / np.pi * h

    map_x = j2.astype(np.float32)
    map_y = i2.astype(np.float32)
    return cv2.remap(img, map_x, map_y,
                     interpolation=cv2.INTER_LINEAR,
                     borderMode=cv2.BORDER_WRAP)

def process_single_image(file_path, output_base, pitch_deg):
    """Process a single image with pitch rotation."""
    output_dir = Path(output_base) / str(pitch_deg)
    output_dir.mkdir(parents=True, exist_ok=True)
    output_path = output_dir / file_path.name
    
    if output_path.exists():
        return None # Skip if exists
    
    img = cv2.imread(str(file_path))
    if img is None:
        return f"Error: Could not read {file_path}"
    
    pitched = pitch_rotate_equirect(img, pitch_deg=pitch_deg)
    cv2.imwrite(str(output_path), pitched)
    return f"Processed {file_path.name}"

def generate_pitched_panoramas(source_dir, output_base, angles, max_workers=8):
    """Generate pitched panoramas using multithreading."""
    png_files = list(Path(source_dir).glob("*.png"))
    print(f"Found {len(png_files)} source images.")
    
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = []
        for file in png_files:
            for angle in angles:
                futures.append(executor.submit(process_single_image, file, output_base, angle))
        
        print(f"Generating pitched images for {len(angles)} angles...")
        for future in tqdm(concurrent.futures.as_completed(futures), total=len(futures)):
            result = future.result()
            if result and "Error" in result:
                print(result)

generate_pitched_panoramas(SOURCE_PANOS_DIR, PITCHED_PANOS_DIR, PITCH_ANGLES)

In [ ]:
def run_pitch_transformation(input_base, output_base, angles):
    """Transform pitched panoramas to perspective images."""
    for angle in angles:
        input_dir = os.path.join(input_base, str(angle))
        output_dir = os.path.join(output_base, str(angle))
        
        if not os.path.exists(input_dir):
            print(f"Skipping {angle}°: Input directory not found.")
            continue
            
        print(f"Transforming Pitch {angle}° -> {output_dir}")
        image_transformer = ImageTransformer(dir_input=input_dir, dir_output=output_dir)
        image_transformer.transform_images(
            style_list="perspective",
            FOV=90,
            theta=90,
            phi=0,
            aspects=(10, 10),
            show_size=100
        )
        
        # Cleanup
        os.system(f"find {output_dir} -name '*Direction_0*' -delete")
        os.system(f"find {output_dir} -name '*Direction_180*' -delete")

run_pitch_transformation(PITCHED_PANOS_DIR, TRANSFORMED_DIR, PITCH_ANGLES)

In [ ]:
def search_csv_files(directory, keyword=None):
    csv_files = []
    for root, dirs, files in os.walk(directory):
        for file in files:
            if file.endswith('.csv'):
                if keyword is None or keyword in file:
                    csv_files.append(os.path.join(root, file))
    return csv_files

def load_pitch_results(base_path, angles, sample_df):
    """Load and merge results from different pitch runs."""
    merged_df = sample_df[['pano_id', 'slope_1m', 'slope_10m', 'slope_30m']].copy()
    merged_df = merged_df.set_index('pano_id')
    
    available_ratios = []
    
    for angle in angles:
        # Assuming structure: base_path/{angle}/perspective_degraded/*.csv
        search_path = os.path.join(base_path, str(angle), 'perspective_degraded')
        csv_files = search_csv_files(search_path, keyword='adjust')
        
        if csv_files:
            # Use the first found CSV
            res_df = pd.read_csv(csv_files[0])
            
            # Calculate availability
            n = len(res_df)
            available_ratios.append(n / 2000 * 100) # Assuming 2000 total samples
            
            res_df.set_index('pano_id', inplace=True)
            res_df = res_df[['adjusted_angle']]
            res_df = res_df.rename(columns={'adjusted_angle': f'pitch{angle}_adjusted_angle'})
            
            # Drop duplicates
            res_df = res_df[~res_df.index.duplicated(keep='first')]
            
            merged_df = merged_df.join(res_df, how='left')
            
            # Calculate difference
            merged_df[f'diff_pitch{angle}'] = merged_df[f'pitch{angle}_adjusted_angle'] - merged_df['slope_1m']
        else:
            print(f"No results found for Pitch {angle}°")
            available_ratios.append(0)
            
    return merged_df, available_ratios

# Run analysis
analyzed_df, availability = load_pitch_results(TRANSFORMED_DIR, PITCH_ANGLES, inliers_05deg)
display(analyzed_df.head())

In [ ]:
def plot_pitch_uncertainty(sample_df, availability_pct, angles, save_path=None):
    import matplotlib.ticker as mticker

    # Filter valid angles
    plot_angles = [a for a in angles if f'diff_pitch{a}' in sample_df.columns]
    if not plot_angles:
        print("No data to plot.")
        return

    # Prepare data
    rename_map = {f'diff_pitch{a}': f"{a}°" for a in plot_angles}
    plot_df = (
        sample_df[[f'diff_pitch{a}' for a in plot_angles]]
        .rename(columns=rename_map)
        .melt(var_name='Pitch angle (°)', value_name='Slope difference (°)')
    )
    
    order_labels = [f"{a}°" for a in plot_angles]
    
    # Plotting
    sns.set_theme(style='whitegrid', context='talk')
    fig = plt.figure(figsize=(14, 8))
    gs = fig.add_gridspec(nrows=2, ncols=1, height_ratios=[2, 1], hspace=0.32)
    
    # Boxplot
    ax_box = fig.add_subplot(gs[0])
    sns.boxplot(
        data=plot_df,
        x='Pitch angle (°)',
        y='Slope difference (°)',
        ax=ax_box,
        width=0.55,
        fliersize=2.2,
        linewidth=1.2,
        medianprops={'color': '#0F3460', 'linewidth': 2},
        whiskerprops={'color': '#0F3460', 'linewidth': 1.2},
        capprops={'color': '#0F3460', 'linewidth': 1.2},
        showfliers=False,
        showcaps=False,
        palette=sns.cubehelix_palette(len(order_labels), start=0.35, rot=-0.15, light=0.85, dark=0.25)
    )
    
    ax_box.axhline(0, color='#5B6270', linestyle='--', linewidth=1, alpha=0.8, zorder=0)
    ax_box.set_xlabel('')
    ax_box.set_ylabel('Slope difference (°)', labelpad=12)
    ax_box.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}'))
    
    # Availability Line Plot
    ax_line = fig.add_subplot(gs[1], sharex=ax_box)
    positions = np.arange(len(order_labels))
    
    ax_line.plot(positions, availability_pct, color=sns.color_palette('crest', n_colors=7)[-2], marker='o', linewidth=2.4, label='Availability')
    ax_line.fill_between(positions, availability_pct, color=sns.color_palette('crest', n_colors=7)[-2], alpha=0.15)
    
    ax_line.set_xlabel('Pitch angle (°)', labelpad=12)
    ax_line.set_ylabel('Available imagery (%)', labelpad=12)
    ax_line.set_xticks(positions)
    ax_line.set_xticklabels(order_labels)
    ax_line.set_ylim(min(min(availability_pct), 60) * 0.95, min(105, max(availability_pct) * 1.2))
    
    if save_path:
        fig.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()

# Plot
plot_pitch_uncertainty(
    analyzed_df,
    availability,
    PITCH_ANGLES,
    save_path=FIGURE_SAVE_PATH
)